我这里直接沿用whole_image里PCC后得到的features，然后去掉和shape相关的

In [1]:

# ============================================================
# Step 2: Habitat feature selection
# Two modes are supported:
#   1. reuse_whole_pcc: reuse whole-tumor PCC-selected features, then remove shape features.
#   2. remove_shape_then_rerun_pcc: remove shape features first, then rerun PCC inside the current habitat table.
# ============================================================

import os
import numpy as np
import pandas as pd


In [17]:

# ============================================================
# 1. Settings
# ============================================================

# Choose one:
#   "reuse_whole_pcc"
#   "remove_shape_then_rerun_pcc"
feature_selection_mode = "remove_shape_then_rerun_pcc"

# Used only when feature_selection_mode == "reuse_whole_pcc".
whole_pcc_path = "/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx"

# Change these paths to switch among:
#   habitats avg
#   habitats sum
#   habitats_individual avg
habitat_root = "/host/d/projects/Habitats/radiomics/habitats_individual"

mode = 'avg'  # 'avg' or 'sum'
input_table_name = f"habitat_radiomics_measurements_{mode}_normalized.xlsx"
output_table_name = f"habitat_radiomics_measurements_{mode}_PCC.xlsx"
feature_list_name = f"habitat_PCC_feature_list_without_shape_{mode}.xlsx"

habitat_normalized_path = os.path.join(
    habitat_root,
    input_table_name,
)

habitat_pcc_out_path = os.path.join(
    habitat_root,
    output_table_name,
)

feature_list_out_path = os.path.join(
    habitat_root,
    feature_list_name,
)

# Used only when feature_selection_mode == "remove_shape_then_rerun_pcc".
pcc_threshold = 0.90

non_feature_cols = [
    "Patient_set",
    "Patient_index",
    "Image_filepath",
    "Mask_filepath",
]

valid_modes = ["reuse_whole_pcc", "remove_shape_then_rerun_pcc"]
if feature_selection_mode not in valid_modes:
    raise ValueError(f"feature_selection_mode must be one of {valid_modes}. Got: {feature_selection_mode}")

print("Feature selection mode:", feature_selection_mode)
print("Habitat normalized input:", habitat_normalized_path)
print("Habitat PCC output:", habitat_pcc_out_path)
print("Feature list output:", feature_list_out_path)


Feature selection mode: remove_shape_then_rerun_pcc
Habitat normalized input: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_radiomics_measurements_avg_normalized.xlsx
Habitat PCC output: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_radiomics_measurements_avg_PCC.xlsx
Feature list output: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_PCC_feature_list_without_shape_avg.xlsx


In [18]:

# ============================================================
# 2. Load habitat normalized table and define candidate features
# ============================================================

habitat_df = pd.read_excel(habitat_normalized_path)

print("Habitat normalized table shape:", habitat_df.shape)

missing_non_feature_cols = [
    col for col in non_feature_cols
    if col not in habitat_df.columns
]

if len(missing_non_feature_cols) > 0:
    raise KeyError(f"Missing non-feature columns in habitat table: {missing_non_feature_cols}")

all_habitat_features = [
    col for col in habitat_df.columns
    if col not in non_feature_cols
]

shape_features_in_habitat = [
    col for col in all_habitat_features
    if col.startswith("original_shape_")
]

non_shape_habitat_features = [
    col for col in all_habitat_features
    if col not in shape_features_in_habitat
]

print("Number of all habitat features:", len(all_habitat_features))
print("Number of shape features removed first:", len(shape_features_in_habitat))
print("Number of non-shape habitat features:", len(non_shape_habitat_features))

if len(non_shape_habitat_features) == 0:
    raise RuntimeError("No non-shape habitat features left for feature selection.")


Habitat normalized table shape: (351, 1019)
Number of all habitat features: 1015
Number of shape features removed first: 14
Number of non-shape habitat features: 1001


In [19]:

# ============================================================
# 3. Build final feature list according to selected mode
# ============================================================

removed_pcc_features = []
whole_pcc_features = []
shape_features_from_whole_pcc = []

if feature_selection_mode == "reuse_whole_pcc":
    # --------------------------------------------------------
    # Current original method:
    # use whole-tumor PCC-selected features, then remove shape.
    # --------------------------------------------------------
    whole_pcc_df = pd.read_excel(whole_pcc_path)

    whole_pcc_features = [
        col for col in whole_pcc_df.columns
        if col not in non_feature_cols
    ]

    shape_features_from_whole_pcc = [
        col for col in whole_pcc_features
        if col.startswith("original_shape_")
    ]

    final_features = [
        col for col in whole_pcc_features
        if col not in shape_features_from_whole_pcc
    ]

    print("Whole-tumor PCC table shape:", whole_pcc_df.shape)
    print("Number of whole-tumor PCC features:", len(whole_pcc_features))
    print("Number of shape features removed from whole-PCC list:", len(shape_features_from_whole_pcc))
    print("Number of final habitat features before habitat-table check:", len(final_features))

elif feature_selection_mode == "remove_shape_then_rerun_pcc":
    # --------------------------------------------------------
    # New method:
    # remove all shape features first, then rerun PCC using the
    # current habitat normalized table itself.
    # --------------------------------------------------------
    X = habitat_df[non_shape_habitat_features].copy()

    # Make sure correlation is calculated on numeric columns only.
    non_numeric_features = [
        col for col in X.columns
        if not pd.api.types.is_numeric_dtype(X[col])
    ]

    if len(non_numeric_features) > 0:
        print("Warning: non-numeric features will be removed before PCC:", len(non_numeric_features))
        for f in non_numeric_features:
            print(" ", f)

    numeric_features = [
        col for col in X.columns
        if col not in non_numeric_features
    ]

    X = X[numeric_features].astype(float)

    # Drop features that are entirely NaN or constant after normalization.
    all_nan_features = [col for col in X.columns if X[col].isna().all()]
    constant_features = [
        col for col in X.columns
        if col not in all_nan_features and X[col].nunique(dropna=True) <= 1
    ]

    pre_pcc_removed_features = non_numeric_features + all_nan_features + constant_features

    if len(pre_pcc_removed_features) > 0:
        print("Features removed before PCC because they are non-numeric/all-NaN/constant:", len(pre_pcc_removed_features))
        for f in pre_pcc_removed_features:
            print(" ", f)

    X = X.drop(columns=[col for col in all_nan_features + constant_features if col in X.columns])

    if X.shape[1] == 0:
        raise RuntimeError("No numeric nonconstant features left for PCC.")

    corr_matrix = X.corr(method="pearson").abs()

    # Upper triangle without the diagonal.
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    removed_pcc_features = [
        col for col in upper.columns
        if any(upper[col] > pcc_threshold)
    ]

    final_features = [
        col for col in X.columns
        if col not in removed_pcc_features
    ]

    # Keep all non-PCC pre-removals in the removed list for documentation.
    removed_pcc_features = pre_pcc_removed_features + removed_pcc_features

    print("PCC threshold:", pcc_threshold)
    print("Number of features entering PCC:", X.shape[1])
    print("Number of features removed by PCC/pre-PCC checks:", len(removed_pcc_features))
    print("Number of final habitat features:", len(final_features))

if len(final_features) == 0:
    raise RuntimeError("No final features selected.")

# Ensure final features exist in this habitat table.
missing_features = [
    col for col in final_features
    if col not in habitat_df.columns
]

if len(missing_features) > 0:
    print("Missing selected features in habitat table:", len(missing_features))
    for f in missing_features:
        print(" ", f)
    raise KeyError("Some final selected features are missing from habitat normalized table.")

print("All final selected features are present in habitat table.")
print("Final feature number:", len(final_features))


PCC threshold: 0.9
Number of features entering PCC: 1001
Number of features removed by PCC/pre-PCC checks: 748
Number of final habitat features: 253
All final selected features are present in habitat table.
Final feature number: 253


In [20]:

# ============================================================
# 4. Save feature list and apply feature selection
# ============================================================

feature_list_df = pd.DataFrame({
    "feature_name": final_features,
})

removed_shape_df = pd.DataFrame({
    "removed_shape_feature": shape_features_in_habitat,
})

removed_pcc_df = pd.DataFrame({
    "removed_pcc_or_pre_pcc_feature": removed_pcc_features,
})

whole_pcc_feature_df = pd.DataFrame({
    "whole_pcc_feature": whole_pcc_features,
})

whole_pcc_removed_shape_df = pd.DataFrame({
    "removed_shape_feature_from_whole_pcc": shape_features_from_whole_pcc,
})

settings_df = pd.DataFrame([
    {
        "feature_selection_mode": feature_selection_mode,
        "whole_pcc_path": whole_pcc_path,
        "habitat_normalized_path": habitat_normalized_path,
        "habitat_pcc_out_path": habitat_pcc_out_path,
        "feature_list_out_path": feature_list_out_path,
        "pcc_threshold": pcc_threshold,
        "n_all_habitat_features": len(all_habitat_features),
        "n_removed_shape_features_in_habitat": len(shape_features_in_habitat),
        "n_final_features": len(final_features),
        "n_removed_pcc_or_pre_pcc_features": len(removed_pcc_features),
    }
])

with pd.ExcelWriter(feature_list_out_path, engine="openpyxl") as writer:
    feature_list_df.to_excel(writer, sheet_name="final_features", index=False)
    removed_shape_df.to_excel(writer, sheet_name="removed_shape_features", index=False)
    removed_pcc_df.to_excel(writer, sheet_name="removed_pcc_features", index=False)
    whole_pcc_feature_df.to_excel(writer, sheet_name="whole_pcc_features", index=False)
    whole_pcc_removed_shape_df.to_excel(writer, sheet_name="whole_pcc_removed_shape", index=False)
    settings_df.to_excel(writer, sheet_name="settings", index=False)

print("Saved final feature list:", feature_list_out_path)

habitat_pcc_df = habitat_df[
    non_feature_cols + final_features
].copy()

print("Final habitat PCC table shape:", habitat_pcc_df.shape)

habitat_pcc_df.to_excel(
    habitat_pcc_out_path,
    index=False,
)

print("Saved:", habitat_pcc_out_path)


Saved final feature list: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_PCC_feature_list_without_shape_avg.xlsx
Final habitat PCC table shape: (351, 257)
Saved: /host/d/projects/Habitats/radiomics/habitats_individual/habitat_radiomics_measurements_avg_PCC.xlsx
